In [ ]:
# path setup: this notebook now lives in revision_scripts/, while the data paths (data/...) and the
# config / netmedgpt imports are relative to the repository root. Safe to re-run; delete if the
# notebook is moved back to the root.
import os, sys
if os.path.basename(os.getcwd()) == 'revision_scripts':
    os.chdir('..')
for p in (os.getcwd(), os.path.join(os.getcwd(), 'revision_scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)


# Walk coverage analysis

How much of the knowledge graph is preserved in the NetMedGPT walk corpus (pseudo-sentences), and what is lost.

Measured on the cached corpus of `build_corpus.py` (`data/new_analysis/corpus`), i.e. exactly the walks NetMedGPT is trained on.
Reverse edges in the split files (`rev_<relation>`) get the token of `<relation>`, as in the walks.
Coverage is measured against the **training graph**, because the walks are generated on training edges only; val/test edges are excluded by design and are reported separately as held-out.

| Measure | Meaning |
|---|---|
| node coverage | KG nodes that appear in the walks (overall and per node type) |
| edge coverage | training edges traversed by at least one walk (overall and per relation). *directed*: (x, y) appears as consecutive nodes; *undirected*: (x, y) or (y, x) appears; *exact*: (x, relation, y) appears with the correct relation token |
| coverage by degree | node coverage and neighbourhood completeness per degree bin (neighbourhood completeness of u = fraction of u's training neighbours seen next to u); edge coverage by the degrees of both end nodes (hub-hub edges) |
| degree preservation | Spearman correlation between node frequency in the walks and node degree |
| metapath coverage | 2-hop relation patterns (r1 → r2) of the training graph that appear in the walks, and Spearman correlation between their walk frequency and their number of paths in the graph |
| relation tokens | walk steps whose relation token is not a training relation of that node pair (a node pair with several relations keeps a single token, `edge_to_rel` in `sentence_generation.py`), and whether such a token is a val/test relation of the pair |
| relation direction | training edges whose reverse edge carries the same relation token |

Outputs (`data/new_analysis/coverage/`):
- `{inference}_seed{seed}/`: `summary.csv`, `node_type_coverage.csv`, `relation_coverage.csv`, `degree_coverage.csv`, `edge_degree_coverage.csv`, `metapath_coverage.csv`, `relation_token_check.csv`
- `summary_all_seeds.csv`, `summary_mean_std.csv`, `relation_coverage_all_seeds.csv`, `degree_coverage_all_seeds.csv` (combined from every `{inference}_seed{seed}` folder in the output folder)

Further sections: **Check 1** (coverage predicted by random-walk theory vs measured) and **Check 3+** (recovery of never-walked edges by the trained model; needs a GPU).

Run the notebook from the repository root (it imports `common.py` and `config.py`).

In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from torch_geometric.nn import Node2Vec
from tqdm import tqdm
from IPython.display import display

from common import load_split, build_corpus
from config import set_seed

## Settings

In [2]:
INFERENCES = ["random_link_split", "zero_shot_split"]   # random_link_split, zero_shot_split, rare_disease or a disease area
# INFERENCES = ["random_link_split"]   # random_link_split, zero_shot_split, rare_disease or a disease area

SEEDS = [1, 2, 3, 4, 5]
# SEEDS = [1]

OUT_DIR = "data/new_analysis/coverage"

DEGREE_BINS = [1, 2, 6, 21, 101, 1001, np.inf]
DEGREE_LABELS = ["1", "2-5", "6-20", "21-100", "101-1000", ">1000"]
CHUNK = 1000000

## Key helpers

In [3]:
# Key helpers: a node pair (x, y) is stored as one int64 key, a triple (x, relation, y) as pair_key * n_rel + relation

def pair_key(x, y, n_nodes):
    return np.asarray(x, dtype=np.int64) * n_nodes + np.asarray(y, dtype=np.int64)


# position of every key in sorted_keys, -1 if the key is not there
def lookup(sorted_keys, keys):
    keys = np.asarray(keys, dtype=np.int64)
    if len(sorted_keys) == 0:
        return np.full(keys.shape, -1, dtype=np.int64)
    idx = np.minimum(np.searchsorted(sorted_keys, keys), len(sorted_keys) - 1)
    return np.where(sorted_keys[idx] == keys, idx, -1)

## Training graph and KG lookups

In [4]:
# Training graph and KG lookups

def build_graph(split):
    nodes, edge = split['nodes'], split['edge']
    n_nodes = int(nodes['node_index'].max()) + 1

    # relation id = z_index - z_min (relation tokens are consecutive)
    rel2z = edge[['relation', 'z_index']].drop_duplicates().set_index('relation')['z_index'].to_dict()
    z_min = int(min(rel2z.values()))
    n_rel = len(rel2z)
    if max(rel2z.values()) - z_min + 1 != n_rel:
        raise ValueError("relation tokens (z_index) are not consecutive")
    relations = [None] * n_rel
    for rel, z in rel2z.items():
        relations[int(z) - z_min] = rel

    unknown = {}

    # the split files store the reverse direction as rev_<relation>; the walks use the same token for both directions
    def triples_of(df, name):
        r = df['relation'].str.replace(r'^rev_', '', regex=True).map(rel2z)
        unknown[name] = int(r.isna().sum())
        df, r = df[r.notna()], r[r.notna()]
        return pair_key(df['x_index'].to_numpy(), df['y_index'].to_numpy(), n_nodes) * n_rel + (r.to_numpy(np.int64) - z_min)

    # training triples, directed pairs and undirected pairs
    triple_keys = np.unique(triples_of(split['train_txgnn'], 'train'))
    t_pair, t_rel = triple_keys // n_rel, triple_keys % n_rel
    t_x, t_y = t_pair // n_nodes, t_pair % n_nodes
    pairs, pair_n_rel = np.unique(t_pair, return_counts=True)
    t_pair_idx = lookup(pairs, t_pair)

    px, py = pairs // n_nodes, pairs % n_nodes
    und_of_pair = pair_key(np.minimum(px, py), np.maximum(px, py), n_nodes)
    und_pairs = np.unique(und_of_pair)
    pair_und_idx = lookup(und_pairs, und_of_pair)
    ua, ub = und_pairs // n_nodes, und_pairs % n_nodes
    not_loop = ua != ub
    degree = np.bincount(ua, minlength=n_nodes) + np.bincount(ub[not_loop], minlength=n_nodes)

    # reverse training edge with the same relation token
    t_reverse_same = lookup(triple_keys, pair_key(t_y, t_x, n_nodes) * n_rel + t_rel) >= 0

    # val/test triples in both directions (split files store one direction)
    held = []
    for name in ['val_txgnn', 'test_txgnn']:
        df = split[name]
        held.append(triples_of(df, name))
        held.append(triples_of(df.rename(columns={'x_index': 'y_index', 'y_index': 'x_index'}), name + '_reversed'))
    heldout_keys = np.unique(np.concatenate(held))

    # full KG: relation token per node pair as in sentence_generation.py (dict built row by row, so the last row wins)
    kg = pd.DataFrame({'key': pair_key(edge['x_index'].to_numpy(), edge['y_index'].to_numpy(), n_nodes),
                       'z': edge['z_index'].to_numpy(np.int64)})
    kg_rel_per_pair = kg.groupby('key')['z'].nunique()
    kg_last = kg.drop_duplicates('key', keep='last').sort_values('key')

    # number of training edges per node and relation, for the 2-hop relation patterns
    out_cnt = np.bincount(t_x * n_rel + t_rel, minlength=n_nodes * n_rel).reshape(n_nodes, n_rel)
    in_cnt = np.bincount(t_y * n_rel + t_rel, minlength=n_nodes * n_rel).reshape(n_nodes, n_rel)
    graph_paths = in_cnt.T.astype(np.float64) @ out_cnt.astype(np.float64)   # [r1, r2]: paths u -r1-> v -r2-> w

    node_type = np.full(n_nodes, None, dtype=object)
    node_type[nodes['node_index'].to_numpy()] = nodes['node_type'].to_numpy()
    rel_node_types = edge.groupby('relation')['node_types'].agg(lambda s: ';'.join(sorted(s.unique()))).to_dict()

    return dict(
        n_nodes=n_nodes, n_kg_nodes=len(nodes), n_rel=n_rel, z_min=z_min, relations=relations,
        rel_node_types=rel_node_types, node_type=node_type,
        triple_keys=triple_keys, t_pair=t_pair, t_rel=t_rel, t_pair_idx=t_pair_idx, t_reverse_same=t_reverse_same,
        pairs=pairs, pair_n_rel=pair_n_rel, pair_und_idx=pair_und_idx,
        und_pairs=und_pairs, ua=ua, ub=ub, degree=degree,
        heldout_keys=heldout_keys, n_val=len(split['val_txgnn']), n_test=len(split['test_txgnn']),
        unknown_relation_rows=unknown,
        kg_pairs=kg_last['key'].to_numpy(), kg_pair_z=kg_last['z'].to_numpy(),
        kg_multi_pairs=int((kg_rel_per_pair > 1).sum()),
        graph_paths=graph_paths,
    )

## Walk statistics

In [5]:
# Walk statistics, accumulated chunk by chunk.
# walks: [N, 2L-1] = [n0, r01, n1, r12, n2, ...] (same layout as node2vec_walks of sentence_generation)

class WalkStats:
    def __init__(self, g):
        self.g = g
        self.node_count = np.zeros(g['n_nodes'], dtype=np.int64)
        self.pair_trav = np.zeros(len(g['pairs']), dtype=np.int64)
        self.triple_trav = np.zeros(len(g['triple_keys']), dtype=np.int64)
        self.rel_count = np.zeros(g['n_rel'], dtype=np.int64)
        self.meta = np.zeros(g['n_rel'] ** 2, dtype=np.int64)
        self.n_walks = 0
        self.n_steps = 0
        self.steps_off_train = 0       # step between two nodes that are not a training pair
        self.steps_wrong_token = 0     # training pair, but the token is not a training relation of the pair
        self.steps_heldout_token = 0   # of these: the token is a val/test relation of the pair
        self.wrong = {}                # triple key -> number of steps, for steps_wrong_token

    def add(self, walks):
        g = self.g
        n_nodes, n_rel = g['n_nodes'], g['n_rel']
        node_tok = walks[:, 0::2]
        rel_tok = walks[:, 1::2] - g['z_min']
        x, y, r = node_tok[:, :-1].ravel(), node_tok[:, 1:].ravel(), rel_tok.ravel()

        self.n_walks += walks.shape[0]
        self.n_steps += len(x)
        self.node_count += np.bincount(node_tok.ravel(), minlength=n_nodes)
        self.rel_count += np.bincount(r, minlength=n_rel)

        pk = pair_key(x, y, n_nodes)
        keys, cnt = np.unique(pk, return_counts=True)
        idx = lookup(g['pairs'], keys)
        self.pair_trav[idx[idx >= 0]] += cnt[idx >= 0]
        self.steps_off_train += int(cnt[idx < 0].sum())

        tkeys, tcnt = np.unique(pk * n_rel + r, return_counts=True)
        tidx = lookup(g['triple_keys'], tkeys)
        exact = tidx >= 0
        self.triple_trav[tidx[exact]] += tcnt[exact]
        wrong = ~exact & (lookup(g['pairs'], tkeys // n_rel) >= 0)
        held = wrong & (lookup(g['heldout_keys'], tkeys) >= 0)
        self.steps_wrong_token += int(tcnt[wrong].sum())
        self.steps_heldout_token += int(tcnt[held].sum())
        for k, c in zip(tkeys[wrong].tolist(), tcnt[wrong].tolist()):
            self.wrong[k] = self.wrong.get(k, 0) + c

        for i in range(rel_tok.shape[1] - 1):
            self.meta += np.bincount(rel_tok[:, i] * n_rel + rel_tok[:, i + 1], minlength=n_rel ** 2)

## Coverage tables

In [6]:
# Coverage tables

def coverage_tables(g, st):
    rows = []

    def add(section, metric, value):
        rows.append({'section': section, 'metric': metric, 'value': float(value)})

    n_rel, relations = g['n_rel'], g['relations']
    degree = g['degree']
    in_graph = degree > 0
    seen = st.node_count > 0

    add('corpus', 'walks', st.n_walks)
    add('corpus', 'walk_steps', st.n_steps)
    for name, n in g['unknown_relation_rows'].items():
        if n:
            add('corpus', f'{name}_rows_with_unknown_relation_excluded', n)

    # nodes
    add('nodes', 'kg_nodes', g['n_kg_nodes'])
    add('nodes', 'training_graph_nodes', in_graph.sum())
    add('nodes', 'covered_nodes', seen.sum())
    add('nodes', 'covered_pct_of_kg_nodes', 100 * seen.sum() / g['n_kg_nodes'])
    add('nodes', 'covered_pct_of_training_graph_nodes', 100 * (seen & in_graph).sum() / in_graph.sum())

    nt = pd.DataFrame({'node_type': g['node_type'], 'in_graph': in_graph, 'covered': seen, 'occurrences': st.node_count})
    nt = nt[nt['node_type'].notna()]
    node_type_df = nt.groupby('node_type').agg(kg_nodes=('in_graph', 'size'), training_graph_nodes=('in_graph', 'sum'),
                                               covered_nodes=('covered', 'sum'), mean_occurrences=('occurrences', 'mean')).reset_index()
    node_type_df['covered_pct_of_kg_nodes'] = 100 * node_type_df['covered_nodes'] / node_type_df['kg_nodes']
    node_type_df['covered_pct_of_training_graph_nodes'] = (100 * node_type_df['covered_nodes']
                                                           / node_type_df['training_graph_nodes'].replace(0, np.nan))

    # edges
    cov_dir = st.pair_trav > 0
    cov_und = np.zeros(len(g['und_pairs']), dtype=bool)
    cov_und[g['pair_und_idx'][cov_dir]] = True
    cov_exact = st.triple_trav > 0

    add('edges', 'kg_directed_pairs', len(g['kg_pairs']))
    add('edges', 'heldout_val_edges_excluded_by_design', g['n_val'])
    add('edges', 'heldout_test_edges_excluded_by_design', g['n_test'])
    add('edges', 'training_triples', len(g['triple_keys']))
    add('edges', 'training_directed_pairs', len(g['pairs']))
    add('edges', 'training_undirected_pairs', len(g['und_pairs']))
    add('edges', 'directed_pair_coverage_pct', 100 * cov_dir.mean())
    add('edges', 'undirected_pair_coverage_pct', 100 * cov_und.mean())
    add('edges', 'exact_triple_coverage_pct', 100 * cov_exact.mean())
    add('edges', 'mean_traversals_per_directed_pair', st.pair_trav.mean())
    add('edges', 'median_traversals_per_directed_pair', np.median(st.pair_trav))

    # per relation
    t_pair_idx = g['t_pair_idx']
    rd = pd.DataFrame({'rel': g['t_rel'], 'dir': cov_dir[t_pair_idx], 'und': cov_und[g['pair_und_idx'][t_pair_idx]],
                       'exact': cov_exact, 'trav': st.triple_trav, 'rev_same': g['t_reverse_same']})
    relation_df = rd.groupby('rel').agg(training_triples=('dir', 'size'), directed_coverage_pct=('dir', 'mean'),
                                        undirected_coverage_pct=('und', 'mean'), exact_coverage_pct=('exact', 'mean'),
                                        mean_traversals=('trav', 'mean'), median_traversals=('trav', 'median'),
                                        reverse_with_same_token_pct=('rev_same', 'mean'))
    for c in ['directed_coverage_pct', 'undirected_coverage_pct', 'exact_coverage_pct', 'reverse_with_same_token_pct']:
        relation_df[c] *= 100
    relation_df = relation_df.reindex(range(n_rel))
    relation_df.insert(0, 'relation', relations)
    relation_df.insert(1, 'node_types', [g['rel_node_types'].get(r, '') for r in relations])
    relation_df['share_of_training_triples_pct'] = 100 * relation_df['training_triples'] / len(g['triple_keys'])
    relation_df['share_of_walk_steps_pct'] = 100 * st.rel_count / max(st.n_steps, 1)
    relation_df = relation_df.reset_index(drop=True)

    add('relation_direction', 'training_triples_with_reverse_same_token_pct', 100 * g['t_reverse_same'].mean())

    # by degree: nodes
    ua, ub = g['ua'], g['ub']
    not_loop = ua != ub
    inc_cov = (np.bincount(ua, weights=cov_und, minlength=g['n_nodes'])
               + np.bincount(ub[not_loop], weights=cov_und[not_loop], minlength=g['n_nodes']))
    completeness = np.where(in_graph, inc_cov / np.maximum(degree, 1), np.nan)

    nd = pd.DataFrame({'degree_bin': pd.cut(degree, bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS),
                       'covered': seen, 'completeness': completeness, 'occurrences': st.node_count, 'degree': degree})[in_graph]
    degree_df = nd.groupby('degree_bin', observed=False).agg(
        nodes=('covered', 'size'), mean_degree=('degree', 'mean'), node_coverage_pct=('covered', 'mean'),
        mean_neighbourhood_completeness_pct=('completeness', 'mean'),
        median_neighbourhood_completeness_pct=('completeness', 'median'),
        mean_occurrences=('occurrences', 'mean')).reset_index()
    for c in ['node_coverage_pct', 'mean_neighbourhood_completeness_pct', 'median_neighbourhood_completeness_pct']:
        degree_df[c] *= 100

    add('degree', 'mean_neighbourhood_completeness_pct', 100 * np.nanmean(completeness))
    add('degree', 'spearman_node_occurrences_vs_degree', spearmanr(st.node_count[in_graph], degree[in_graph]).correlation)

    # by degree: edges, binned by the degree of both end nodes
    da, db = degree[ua], degree[ub]
    ed = pd.DataFrame({
        'min_endpoint_degree': pd.cut(np.minimum(da, db), bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS),
        'max_endpoint_degree': pd.cut(np.maximum(da, db), bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS),
        'covered': cov_und})
    edge_degree_df = ed.groupby(['min_endpoint_degree', 'max_endpoint_degree'], observed=True).agg(
        undirected_pairs=('covered', 'size'), coverage_pct=('covered', 'mean')).reset_index()
    edge_degree_df['coverage_pct'] *= 100

    # 2-hop relation patterns
    gp = g['graph_paths']
    wc = st.meta.reshape(n_rel, n_rel)
    i, j = np.nonzero((gp > 0) | (wc > 0))
    metapath_df = pd.DataFrame({'relation_1': [relations[k] for k in i], 'relation_2': [relations[k] for k in j],
                                'graph_paths': gp[i, j], 'walk_count': wc[i, j]})
    metapath_df['in_training_graph'] = metapath_df['graph_paths'] > 0
    metapath_df['observed_in_walks'] = metapath_df['walk_count'] > 0
    in_g = metapath_df[metapath_df['in_training_graph']]
    add('metapaths', 'two_hop_patterns_in_training_graph', len(in_g))
    add('metapaths', 'two_hop_patterns_observed', in_g['observed_in_walks'].sum())
    add('metapaths', 'two_hop_pattern_coverage_pct', 100 * in_g['observed_in_walks'].mean())
    add('metapaths', 'two_hop_patterns_in_walks_not_in_training_graph', (~metapath_df['in_training_graph']).sum())
    add('metapaths', 'spearman_walk_count_vs_graph_paths', spearmanr(in_g['walk_count'], in_g['graph_paths']).correlation)

    # relation tokens
    add('relation_tokens', 'kg_pairs_with_multiple_relations', g['kg_multi_pairs'])
    add('relation_tokens', 'training_pairs_with_multiple_relations', (g['pair_n_rel'] > 1).sum())
    add('relation_tokens', 'walk_steps_off_training_pairs', st.steps_off_train)
    add('relation_tokens', 'walk_steps_token_not_training_relation', st.steps_wrong_token)
    add('relation_tokens', 'walk_steps_token_not_training_relation_pct', 100 * st.steps_wrong_token / max(st.n_steps, 1))
    add('relation_tokens', 'walk_steps_token_is_heldout_relation', st.steps_heldout_token)

    check = []
    for k, c in st.wrong.items():
        pk, r = divmod(k, n_rel)
        x, y = divmod(pk, g['n_nodes'])
        lo, hi = np.searchsorted(g['t_pair'], [pk, pk + 1])
        check.append({'x_index': x, 'y_index': y, 'token_relation': relations[r],
                      'training_relations': ';'.join(relations[t] for t in g['t_rel'][lo:hi]),
                      'token_is_heldout_relation': bool(lookup(g['heldout_keys'], [k])[0] >= 0),
                      'walk_steps': c})
    token_check_df = pd.DataFrame(check, columns=['x_index', 'y_index', 'token_relation', 'training_relations',
                                                  'token_is_heldout_relation', 'walk_steps'])

    tables = dict(node_type_coverage=node_type_df, relation_coverage=relation_df, degree_coverage=degree_df,
                  edge_degree_coverage=edge_degree_df, metapath_coverage=metapath_df, relation_token_check=token_check_df)
    return pd.DataFrame(rows), tables

## Walk generation

In [7]:
# Walk generation: relation tokens inserted into Node2Vec walks as in sentence_generation.py
# (used by walk_coverage_curated_kg.ipynb to generate new walks)

# vectorised add_relations_to_walk (sentence_generation.py): a walk is dropped if one of its steps has no relation
def add_relations(rw, g):
    idx = lookup(g['kg_pairs'], pair_key(rw[:, :-1], rw[:, 1:], g['n_nodes']))
    keep = (idx >= 0).all(axis=1)
    out = np.empty((int(keep.sum()), 2 * rw.shape[1] - 1), dtype=np.int64)
    out[:, 0::2] = rw[keep]
    out[:, 1::2] = g['kg_pair_z'][idx[keep]]
    return out


## Combine runs

In [8]:
# Combine every {inference}_seed{seed} folder of the output folder

def combine_runs(out_dir):
    collected = {'summary': [], 'relation_coverage': [], 'degree_coverage': []}
    for run_dir in sorted(glob.glob(os.path.join(out_dir, "*_seed*"))):
        m = re.match(r"(.+)_seed(\d+)$", os.path.basename(run_dir))
        if not m or not os.path.isdir(run_dir):
            continue
        for name in collected:
            path = os.path.join(run_dir, f"{name}.csv")
            if os.path.exists(path):
                df = pd.read_csv(path)
                df.insert(0, 'seed', int(m.group(2)))
                df.insert(0, 'inference', m.group(1))
                collected[name].append(df)

    if not collected['summary']:
        return
    summary = pd.concat(collected['summary'], ignore_index=True)
    summary.to_csv(os.path.join(out_dir, "summary_all_seeds.csv"), index=False)
    (summary.groupby(['inference', 'section', 'metric'], sort=False)['value']
            .agg(['mean', 'std', 'count']).reset_index()
            .to_csv(os.path.join(out_dir, "summary_mean_std.csv"), index=False))
    for name in ['relation_coverage', 'degree_coverage']:
        if collected[name]:
            pd.concat(collected[name], ignore_index=True).to_csv(os.path.join(out_dir, f"{name}_all_seeds.csv"), index=False)
    print(f"Combined results saved to {out_dir}")

## Run: coverage of the cached corpus

In [9]:
os.makedirs(OUT_DIR, exist_ok=True)
results = {}

for inference in INFERENCES:
    for seed in SEEDS:
        print(f"\n===== {inference} | seed {seed} =====")
        split = load_split(inference, seed)
        data = build_corpus(split)
        g = build_graph(split)

        # coverage of the cached corpus NetMedGPT is trained on
        walks = data['node2vec_walks'].numpy()
        st = WalkStats(g)
        for i in tqdm(range(0, len(walks), CHUNK), desc="corpus"):
            st.add(walks[i:i + CHUNK])
        summary, tables = coverage_tables(g, st)
        results[(inference, seed)] = (summary, tables)

        run_dir = os.path.join(OUT_DIR, f"{inference}_seed{seed}")
        os.makedirs(run_dir, exist_ok=True)
        summary.to_csv(os.path.join(run_dir, "summary.csv"), index=False)
        for name, df in tables.items():
            df.to_csv(os.path.join(run_dir, f"{name}.csv"), index=False)
        print(f"Results saved to {run_dir}")

        del split, data, g, st, walks

combine_runs(OUT_DIR)


===== random_link_split | seed 1 =====
Reproducibility settings
  seed                     : 1
  torch.manual_seed        : 1
  torch.cuda.manual_seed   : 1
  numpy / random seed      : 1
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed1.pt


corpus: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.06s/it]


Results saved to data/result/new_analysis/walk_coverage/random_link_split_seed1

===== random_link_split | seed 2 =====
Reproducibility settings
  seed                     : 2
  torch.manual_seed        : 2
  torch.cuda.manual_seed   : 2
  numpy / random seed      : 2
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


... (31 outputs left out)

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/zero_shot_split_seed5.pt


corpus: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.06s/it]


Results saved to data/result/new_analysis/walk_coverage/zero_shot_split_seed5
Combined results saved to data/result/new_analysis/walk_coverage


## Summary over seeds (mean / std)

In [10]:
summary_mean_std = pd.read_csv(os.path.join(OUT_DIR, "summary_mean_std.csv"))
pd.set_option("display.max_rows", 200)
display(summary_mean_std.pivot_table(index=["section", "metric"], columns="inference", values="mean", sort=False).round(4))

inference                                                           random_link_split  \
section            metric                                                               
corpus             walks                                                 3.684739e+06   
                   walk_steps                                            1.473896e+07   
nodes              kg_nodes                                              1.293750e+05   
                   training_graph_nodes                                  1.232188e+05   
                   covered_nodes                                         1.231734e+05   
                   covered_pct_of_kg_nodes                               9.520650e+01   
                   covered_pct_of_training_graph_nodes                   9.996320e+01   
edges              kg_directed_pairs                                     8.099284e+06   
                   heldout_val_edges_excluded_by_design                  9.619380e+05   
                   heldout_test_edges_excluded_by_design                 4.050240e+05   
                   training_triples                                      6.697848e+06   
                   training_directed_pairs                               6.697424e+06   
                   training_undirected_pairs                             3.366342e+06   
                   directed_pair_coverage_pct                            5.904730e+01   
                   undirected_pair_coverage_pct                          7.735090e+01   
                   exact_triple_coverage_pct                             5.904250e+01   
                   mean_traversals_per_directed_pair                     2.200700e+00   
                   median_traversals_per_directed_pair                   1.000000e+00   
relation_direction training_triples_with_reverse_same_token_pct          9.947090e+01   
degree             mean_neighbourhood_completeness_pct                   9.713640e+01   
                   spearman_node_occurrences_vs_degree                   6.053000e-01   
metapaths          two_hop_patterns_in_training_graph                    2.590000e+02   
                   two_hop_patterns_observed                             2.572000e+02   
                   two_hop_pattern_coverage_pct                          9.930500e+01   
                   two_hop_patterns_in_walks_not_in_training_graph       0.000000e+00   
                   spearman_walk_count_vs_graph_paths                    8.719000e-01   
relation_tokens    kg_pairs_with_multiple_relations                      8.420000e+02   
                   training_pairs_with_multiple_relations                4.236000e+02   
                   walk_steps_off_training_pairs                         0.000000e+00   
                   walk_steps_token_not_training_relation                3.480000e+02   
                   walk_steps_token_not_training_relation_pct            2.400000e-03   
                   walk_steps_token_is_heldout_relation                  3.480000e+02   

inference                                                           zero_shot_split  
section            metric                                                            
corpus             walks                                               3.685520e+06  
                   walk_steps                                          1.474208e+07  
nodes              kg_nodes                                            1.293750e+05  
                   training_graph_nodes                                1.232134e+05  
                   covered_nodes                                       1.231656e+05  
                   covered_pct_of_kg_nodes                             9.520050e+01  
                   covered_pct_of_training_graph_nodes                 9.996120e+01  
edges              kg_directed_pairs                                   8.099284e+06  
                   heldout_val_edges_excluded_by_design                9.619420e+05  
                   heldout_

## Coverage per relation (mean over seeds)

In [11]:
relation_all = pd.read_csv(os.path.join(OUT_DIR, "relation_coverage_all_seeds.csv"))
display(relation_all.groupby(["inference", "relation"], sort=False)
                    [["training_triples", "directed_coverage_pct", "undirected_coverage_pct", "exact_coverage_pct",
                      "mean_traversals", "share_of_training_triples_pct", "share_of_walk_steps_pct"]]
                    .mean().round(2))

training_triples  \
inference         relation                                       
random_link_split anatomy_protein_present            2524012.0   
                  protein_protein                     533786.0   
                  bioprocess_protein                  240738.0   
                  anatomy_protein_absent               33062.0   
                  molfunc_protein                     115594.0   
                  cellcomp_protein                    138656.0   
                  disease_protein                     133682.0   
                  pathway_protein                      70900.0   
                  drug_protein                         42397.2   
                  phenotype_protein                     5536.0   
                  exposure_protein                      2014.0   
                  drug_drug                          2221622.0   
                  drug_effect                         107704.0   
                  contraindication                     25498.0   
                  indication                            7804.0   
                  off-label use                         2135.0   
                  disease_phenotype_positive          249902.0   
                  phenotype_phenotype                  31148.0   
                  disease_phenotype_negative            1982.0   
                  disease_disease                      53522.0   
                  exposure_disease                      3830.0   
                  bioprocess_bioprocess                87924.0   
                  exposure_bioprocess                   2702.0   
                  molfunc_molfunc                      22566.0   
                  exposure_molfunc                        76.0   
                  cellcomp_cellcomp                     8056.0   
                  exposure_cellcomp                       18.0   
                  exposure_exposure                     3440.0   
                  pathway_pathway                       4214.0   
                  anatomy_anatomy                      23328.0   
zero_shot_split   anatomy_protein_present            2524012.0   
                  protein_protein                     533786.0   
                  bioprocess_protein                  240738.0   
                  anatomy_protein_absent               33062.0   
                  molfunc_protein                     115594.0   
                  cellcomp_protein                    138656.0   
                  disease_protein                     133682.0   
                  pathway_protein                      70900.0   
                  drug_protein                         42397.2   
                  phenotype_protein                     5536.0   
                  exposure_protein                      2014.0   
                  drug_drug                          2221622.0   
                  drug_effect                         107704.0   
                  contraindication                     25900.2   
                  indication                            7821.8   
                  off-label use                         4217.6   
                  disease_phenotype_positive          249902.0   
                  phenotype_phenotype                  31148.0   
                  disease_phenotype_negative            1982.0   
                  disease_disease                      53522.0   
                  exposure_disease                      3830.0   
                  bioprocess_bioprocess                87924.0   
                  exposure_bioprocess                   2702.0   
                  molfunc_molfunc                      22566.0   
                  exposure_molfunc                        76.0   
                  cellcomp_cellcomp                     8056.0   
                  exposure_cellcomp                       18.0   
                  exposure_exposure                     3440.0   
                  pathway_pathway                       4214.0   
                  an

## Coverage by node degree (mean over seeds)

In [12]:
degree_all = pd.read_csv(os.path.join(OUT_DIR, "degree_coverage_all_seeds.csv"))
display(degree_all.groupby(["inference", "degree_bin"], sort=False)
                  [["nodes", "mean_degree", "node_coverage_pct", "mean_neighbourhood_completeness_pct", "mean_occurrences"]]
                  .mean().round(2))

nodes  mean_degree  node_coverage_pct  \
inference         degree_bin                                            
random_link_split 1           32501.6         1.00              99.91   
                  2-5         39056.8         3.05              99.97   
                  6-20        20613.8        10.57              99.99   
                  21-100      16422.0        49.85              99.99   
                  101-1000    13677.2       233.05             100.00   
                  >1000         947.4      2488.02             100.00   
zero_shot_split   1           32520.6         1.00              99.91   
                  2-5         39074.6         3.05              99.96   
                  6-20        20587.4        10.57              99.99   
                  21-100      16401.8        49.87              99.99   
                  101-1000    13681.4       233.08             100.00   
                  >1000         947.6      2487.88             100.00   

                              mean_neighbourhood_completeness_pct  \
inference         degree_bin                                        
random_link_split 1                                         99.91   
                  2-5                                       99.90   
                  6-20                                      99.57   
                  21-100                                    95.40   
                  101-1000                                  83.96   
                  >1000                                     55.64   
zero_shot_split   1                                         99.91   
                  2-5                                       99.90   
                  6-20                                      99.58   
                  21-100                                    95.40   
                  101-1000                                  83.80   
                  >1000                                     54.93   

                              mean_occurrences  
inference         degree_bin                    
random_link_split 1                      65.84  
                  2-5                    89.58  
                  6-20                  120.81  
                  21-100                189.72  
                  101-1000              340.39  
                  >1000                2663.64  
zero_shot_split   1                      65.91  
                  2-5                    89.76  
                  6-20                  121.26  
                  21-100                190.28  
                  101-1000              339.64  
                  >1000                2651.77

## Single run: detailed tables

Tables of one split / seed from this session (`results`); saved CSVs are in `OUT_DIR/{inference}_seed{seed}/`.

In [13]:
summary, tables = results[(INFERENCES[0], SEEDS[0])]
for name in ["node_type_coverage", "edge_degree_coverage", "metapath_coverage", "relation_token_check"]:
    print(name)
    display(tables[name])

node_type_coverage


,node_type,kg_nodes,training_graph_nodes,covered_nodes,mean_occurrences,covered_pct_of_kg_nodes,covered_pct_of_training_graph_nodes
0,anatomy,14035,12146,12137,257.793659,86.476665,99.925902
1,biological_process,28642,27873,27872,107.478528,97.311640,99.996412
2,cellular_component,4176,3883,3883,115.094349,92.983716,100.000000
3,disease,17080,16468,16467,127.824297,96.411007,99.993928
4,drug,7957,7467,7433,152.251225,93.414603,99.544663
5,effect/phenotype,15311,14642,14642,123.915877,95.630592,100.000000
6,exposure,818,787,787,100.047677,96.210269,100.000000
7,gene/protein,27671,27227,27224,163.748256,98.384590,99.988982
8,molecular_function,11169,10238,10238,102.674277,91.664428,100.000000
9,pathway,2516,2504,2504,78.932035,99.523052,100.000000


edge_degree_coverage


,min_endpoint_degree,max_endpoint_degree,undirected_pairs,coverage_pct
0,1,1,255,100.000000
1,1,2-5,8255,99.975772
2,1,6-20,7385,99.945836
3,1,21-100,6920,99.783237
4,1,101-1000,7132,99.971957
5,1,>1000,2305,99.609544
6,2-5,2-5,15028,99.960075
7,2-5,6-20,25131,99.928375
8,2-5,21-100,25253,99.754485
9,2-5,101-1000,22091,99.900412


metapath_coverage


,relation_1,relation_2,graph_paths,walk_count,in_training_graph,observed_in_walks
0,anatomy_protein_present,anatomy_protein_present,1.311456e+10,1927337,True,True
1,anatomy_protein_present,protein_protein,4.243636e+07,144844,True,True
2,anatomy_protein_present,bioprocess_protein,8.397019e+06,35683,True,True
3,anatomy_protein_present,anatomy_protein_absent,1.058050e+08,18565,True,True
4,anatomy_protein_present,molfunc_protein,4.018998e+06,18894,True,True
...,...,...,...,...,...,...
254,pathway_pathway,pathway_protein,4.363900e+04,18299,True,True
255,pathway_pathway,pathway_pathway,1.949600e+04,35343,True,True
256,anatomy_anatomy,anatomy_protein_present,3.652762e+06,15441,True,True
257,anatomy_anatomy,anatomy_protein_absent,3.849300e+04,162,True,True


relation_token_check


,x_index,y_index,token_relation,training_relations,token_is_heldout_relation,walk_steps
0,22425,27158,disease_phenotype_positive,disease_phenotype_negative,True,8
1,22488,27603,disease_phenotype_positive,disease_phenotype_negative,True,4
2,22555,27180,disease_phenotype_positive,disease_phenotype_negative,True,9
3,22675,27224,disease_phenotype_positive,disease_phenotype_negative,True,8
4,23139,27402,disease_phenotype_positive,disease_phenotype_negative,True,1
5,23351,27276,disease_phenotype_positive,disease_phenotype_negative,True,5
6,23776,27747,disease_phenotype_positive,disease_phenotype_negative,True,5
7,24168,27183,disease_phenotype_positive,disease_phenotype_negative,True,5
8,24340,27353,disease_phenotype_positive,disease_phenotype_negative,True,9
9,24516,27629,disease_phenotype_positive,disease_phenotype_negative,True,2


# Check 1: theory vs measurement (what is lost, and why)

The walks are a random sample, so which edges they miss can be **predicted from the training graph alone**, without generating any walk.
Every step of a walk (Node2Vec with p = q = 1, as in `sentence_generation.py`) goes to a uniformly chosen training neighbour. From the transition matrix we compute, for every training edge u → v, the expected number of walk steps over it (λ). The edge is missed with probability exp(−λ).

Example: a node with 10 neighbours that is left 30 times misses a given neighbour with probability (9/10)^30 ≈ 4%; a hub with 2,000 neighbours left 2,600 times misses about 27%.

If the predicted and the measured coverage agree, the information loss is fully explained by random sampling: it is known in advance which edges are lost (mainly edges between high-degree nodes).

Outputs per split / seed: `check1_overall.csv`, `check1_degree.csv`, `check1_relation.csv`; combined: `check1_*_all_seeds.csv`.

In [15]:
import scipy.sparse as sp

# Expected number of walk steps over every training pair, computed from the training graph alone (no walks).
# Same walk process as Node2Vec with p = q = 1 in sentence_generation.py: walks_per_node walks start at every node
# 0..max node index, every step goes to a uniformly chosen training neighbour (a pair listed twice counts twice),
# and a walk that reaches a node without outgoing training edges is dropped.
#   a[t][u] : expected number of walks at u after t steps
#   h[k][u] : probability that k more steps can be taken from u
#   lam[u -> v] = sum_t a[t][u] * P[u, v] * h[steps - 1 - t][v]   (kept walks that step from u to v)
# A pair is missed with probability exp(-lam) (Poisson approximation).
def expected_traversals(split, g, walks_per_node, walk_length):
    n = g['n_nodes']
    train = split['train_txgnn']
    x = train['x_index'].to_numpy(np.int64)
    y = train['y_index'].to_numpy(np.int64)
    n_walk_nodes = int(max(x.max(), y.max())) + 1

    keys, mult = np.unique(pair_key(x, y, n), return_counts=True)
    out_w = np.bincount(x, minlength=n).astype(np.float64)
    inv_w = np.divide(1.0, out_w, out=np.zeros(n), where=out_w > 0)
    ku, kv = keys // n, keys % n
    P = sp.csr_matrix((mult * inv_w[ku], (ku, kv)), shape=(n, n))

    steps = walk_length - 1
    h = [np.ones(n)]
    for _ in range(steps):
        h.append(P @ h[-1])
    a = [np.zeros(n)]
    a[0][:n_walk_nodes] = walks_per_node
    for _ in range(steps):
        a.append(P.T @ a[-1])

    idx = lookup(keys, g['pairs'])
    if (idx < 0).any():
        raise ValueError("training pairs missing from the walk graph")
    pu, pv = g['pairs'] // n, g['pairs'] % n
    p_uv = mult[idx] * inv_w[pu]
    lam = np.zeros(len(g['pairs']))
    for t in range(steps):
        lam += a[t][pu] * p_uv * h[steps - 1 - t][pv]

    sampled = a[0].sum()
    kept = (a[0] * h[steps]).sum()
    occurrences = sum(a[t] * h[steps - t] for t in range(steps + 1))
    return lam, dict(expected_sampled_walks=sampled, expected_kept_walks=kept,
                     expected_dropped_walks_pct=100 * (1 - kept / sampled), expected_occurrences=occurrences)


def covered_undirected(g, st):
    cov = np.zeros(len(g['und_pairs']), dtype=bool)
    cov[g['pair_und_idx'][st.pair_trav > 0]] = True
    return cov


# mean of a per-pair value (coverage flag or probability) over every node's undirected training pairs
def per_node_mean(g, und_values):
    ua, ub = g['ua'], g['ub']
    not_loop = ua != ub
    s = (np.bincount(ua, weights=und_values, minlength=g['n_nodes'])
         + np.bincount(ub[not_loop], weights=und_values[not_loop], minlength=g['n_nodes']))
    return np.where(g['degree'] > 0, s / np.maximum(g['degree'], 1), np.nan)


def predicted_coverage(g, lam):
    p_dir = 1 - np.exp(-lam)
    lam_und = np.bincount(g['pair_und_idx'], weights=lam, minlength=len(g['und_pairs']))
    return p_dir, 1 - np.exp(-lam_und)


def theory_vs_measured(g, st, lam, info):
    p_dir, p_und = predicted_coverage(g, lam)
    cov_dir = st.pair_trav > 0
    cov_und = covered_undirected(g, st)
    comp_pred = per_node_mean(g, p_und)
    comp_meas = per_node_mean(g, cov_und.astype(np.float64))
    in_graph = g['degree'] > 0

    overall = pd.DataFrame([
        ('kept_walks', info['expected_kept_walks'], st.n_walks),
        ('dropped_walks_pct', info['expected_dropped_walks_pct'], np.nan),
        ('mean_traversals_per_directed_pair', lam.mean(), st.pair_trav.mean()),
        ('directed_pair_coverage_pct', 100 * p_dir.mean(), 100 * cov_dir.mean()),
        ('undirected_pair_coverage_pct', 100 * p_und.mean(), 100 * cov_und.mean()),
        ('mean_neighbourhood_completeness_pct', 100 * np.nanmean(comp_pred), 100 * np.nanmean(comp_meas)),
        ('spearman_node_occurrences_vs_degree',
         spearmanr(info['expected_occurrences'][in_graph], g['degree'][in_graph]).correlation,
         spearmanr(st.node_count[in_graph], g['degree'][in_graph]).correlation),
    ], columns=['metric', 'predicted', 'measured'])

    nd = pd.DataFrame({'degree_bin': pd.cut(g['degree'], bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS),
                       'pred': comp_pred, 'meas': comp_meas,
                       'occ_pred': info['expected_occurrences'], 'occ_meas': st.node_count})[in_graph]
    degree_df = nd.groupby('degree_bin', observed=False).agg(
        nodes=('pred', 'size'),
        predicted_neighbourhood_completeness_pct=('pred', 'mean'),
        measured_neighbourhood_completeness_pct=('meas', 'mean'),
        predicted_mean_occurrences=('occ_pred', 'mean'),
        measured_mean_occurrences=('occ_meas', 'mean')).reset_index()
    for c in ['predicted_neighbourhood_completeness_pct', 'measured_neighbourhood_completeness_pct']:
        degree_df[c] *= 100

    t_und = g['pair_und_idx'][g['t_pair_idx']]
    rd = pd.DataFrame({'rel': g['t_rel'], 'pred': p_und[t_und], 'meas': cov_und[t_und]})
    relation_df = rd.groupby('rel').agg(training_triples=('pred', 'size'),
                                        predicted_undirected_coverage_pct=('pred', 'mean'),
                                        measured_undirected_coverage_pct=('meas', 'mean'))
    relation_df[['predicted_undirected_coverage_pct', 'measured_undirected_coverage_pct']] *= 100
    relation_df.insert(0, 'relation', [g['relations'][r] for r in relation_df.index])
    return overall, degree_df, relation_df.reset_index(drop=True)


## Check 1: run

In [16]:
check1 = {'overall': [], 'degree': [], 'relation': []}

for inference in INFERENCES:
    for seed in SEEDS:
        print(f"\n===== Check 1 | {inference} | seed {seed} =====")
        split = load_split(inference, seed)
        data = build_corpus(split)
        g = build_graph(split)

        walks = data['node2vec_walks'].numpy()
        st = WalkStats(g)
        for i in range(0, len(walks), CHUNK):
            st.add(walks[i:i + CHUNK])

        param = split['all_param']['node2vec']
        lam, info = expected_traversals(split, g, param['walks_per_node'], param['walk_length'])
        overall, degree_df, relation_df = theory_vs_measured(g, st, lam, info)

        run_dir = os.path.join(OUT_DIR, f"{inference}_seed{seed}")
        os.makedirs(run_dir, exist_ok=True)
        for name, df in [('overall', overall), ('degree', degree_df), ('relation', relation_df)]:
            df.to_csv(os.path.join(run_dir, f"check1_{name}.csv"), index=False)
            df.insert(0, 'seed', seed)
            df.insert(0, 'inference', inference)
            check1[name].append(df)
        print(overall.round(4).to_string(index=False))

        del split, data, g, st, walks, lam

for name, dfs in check1.items():
    pd.concat(dfs, ignore_index=True).to_csv(os.path.join(OUT_DIR, f"check1_{name}_all_seeds.csv"), index=False)
print(f"Check 1 saved to {OUT_DIR}")


===== Check 1 | random_link_split | seed 1 =====
Reproducibility settings
  seed                     : 1
  torch.manual_seed        : 1
  torch.cuda.manual_seed   : 1
  numpy / random seed      : 1
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed1.pt
        inference  seed                              metric    predicted     measured
random_link_split     1                          kept_walks 3690894.3784 3685485.0000
random_link_split     1                   dropped_walks_pct       4.9045          NaN
random_link_split     1   mean_traversals_per_directed_pair       2.2044       2.2011
random_link_split     1          directed_pair_coverage_pct      59.1050      59.0404
random_link_split     1        undirected_pair_coverage_pct      77.4214      77.3492
random_link_split     1 mean_neighbourhood_completeness_pct      97.2251      97.1387
random_link_split     1 spearman_node_occurrences_vs_degree       0.6132       0.6057

===== Check 1 | random_link_split | seed 2 =====
Reproducibility settings
  seed                     : 2
  torch.manual_seed        : 2
  torch.cuda.manual_seed   : 2
  numpy / random seed      : 2
  cudnn.deterministic      : True
  cu

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed2.pt
        inference  seed                              metric    predicted     measured
random_link_split     2                          kept_walks 3691084.7342 3685520.0000
random_link_split     2                   dropped_walks_pct       4.8996          NaN
random_link_split     2   mean_traversals_per_directed_pair       2.2045       2.2012
random_link_split     2          directed_pair_coverage_pct      59.1339      59.0593
random_link_split     2        undirected_pair_coverage_pct      77.4586      77.3935
random_link_split     2 mean_neighbourhood_completeness_pct      97.2258      97.1430
random_link_split     2 spearman_node_occurrences_vs_degree       0.6142       0.6061

===== Check 1 | random_link_split | seed 3 =====
Reproducibility settings
  seed                     : 3
  torch.manual_seed        : 3
  torch.cuda.manual_seed   : 3
  numpy / random seed      : 3
  cudnn.deterministic      : True
  cu

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


... (12 outputs left out)

Loading cached corpus: data/new_analysis/corpus/zero_shot_split_seed4.pt
      inference  seed                              metric    predicted     measured
zero_shot_split     4                          kept_walks 3690506.2832 3684892.0000
zero_shot_split     4                   dropped_walks_pct       4.9145          NaN
zero_shot_split     4   mean_traversals_per_directed_pair       2.2030       2.1997
zero_shot_split     4          directed_pair_coverage_pct      58.9128      58.8297
zero_shot_split     4        undirected_pair_coverage_pct      77.1393      77.0404
zero_shot_split     4 mean_neighbourhood_completeness_pct      97.2077      97.1090
zero_shot_split     4 spearman_node_occurrences_vs_degree       0.6130       0.6045

===== Check 1 | zero_shot_split | seed 5 =====
Reproducibility settings
  seed                     : 5
  torch.manual_seed        : 5
  torch.cuda.manual_seed   : 5
  numpy / random seed      : 5
  cudnn.deterministic      : True
  cudnn.benchmark       

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/zero_shot_split_seed5.pt
      inference  seed                              metric    predicted     measured
zero_shot_split     5                          kept_walks 3690884.9313 3685065.0000
zero_shot_split     5                   dropped_walks_pct       4.9047          NaN
zero_shot_split     5   mean_traversals_per_directed_pair       2.2040       2.2005
zero_shot_split     5          directed_pair_coverage_pct      58.9060      58.8545
zero_shot_split     5        undirected_pair_coverage_pct      77.1249      77.0373
zero_shot_split     5 mean_neighbourhood_completeness_pct      97.2139      97.1243
zero_shot_split     5 spearman_node_occurrences_vs_degree       0.6144       0.6061
Check 1 saved to data/result/new_analysis/walk_coverage


/tmp/ipykernel_1425442/136845210.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(dfs, ignore_index=True).to_csv(os.path.join(OUT_DIR, f"check1_{name}_all_seeds.csv"), index=False)


## Check 1: results (mean over seeds)

In [17]:
for name, keys in [('overall', ['metric']), ('degree', ['degree_bin']), ('relation', ['relation'])]:
    df = pd.read_csv(os.path.join(OUT_DIR, f"check1_{name}_all_seeds.csv"))
    print(name)
    display(df.drop(columns='seed').groupby(['inference'] + keys, sort=False).mean().round(3))

overall


predicted  \
inference         metric                                             
random_link_split kept_walks                           3690367.979   
                  dropped_walks_pct                          4.918   
                  mean_traversals_per_directed_pair          2.204   
                  directed_pair_coverage_pct                59.114   
                  undirected_pair_coverage_pct              77.438   
                  mean_neighbourhood_completeness_pct       97.225   
                  spearman_node_occurrences_vs_degree        0.613   
zero_shot_split   kept_walks                           3691158.118   
                  dropped_walks_pct                          4.898   
                  mean_traversals_per_directed_pair          2.204   
                  directed_pair_coverage_pct                58.922   
                  undirected_pair_coverage_pct              77.146   
                  mean_neighbourhood_completeness_pct       97.212   
                  spearman_node_occurrences_vs_degree        0.614   

                                                          measured  
inference         metric                                            
random_link_split kept_walks                           3684739.000  
                  dropped_walks_pct                            NaN  
                  mean_traversals_per_directed_pair          2.201  
                  directed_pair_coverage_pct                59.047  
                  undirected_pair_coverage_pct              77.351  
                  mean_neighbourhood_completeness_pct       97.136  
                  spearman_node_occurrences_vs_degree        0.605  
zero_shot_split   kept_walks                           3685520.400  
                  dropped_walks_pct                            NaN  
                  mean_traversals_per_directed_pair          2.200  
                  directed_pair_coverage_pct                58.859  
                  undirected_pair_coverage_pct              77.058  
                  mean_neighbourhood_completeness_pct       97.117  
                  spearman_node_occurrences_vs_degree        0.606

degree


nodes  \
inference         degree_bin            
random_link_split 1           32501.6   
                  2-5         39056.8   
                  6-20        20613.8   
                  21-100      16422.0   
                  101-1000    13677.2   
                  >1000         947.4   
zero_shot_split   1           32520.6   
                  2-5         39074.6   
                  6-20        20587.4   
                  21-100      16401.8   
                  101-1000    13681.4   
                  >1000         947.6   

                              predicted_neighbourhood_completeness_pct  \
inference         degree_bin                                             
random_link_split 1                                             99.931   
                  2-5                                           99.926   
                  6-20                                          99.663   
                  21-100                                        95.807   
                  101-1000                                      83.993   
                  >1000                                         55.634   
zero_shot_split   1                                             99.935   
                  2-5                                           99.929   
                  6-20                                          99.688   
                  21-100                                        95.830   
                  101-1000                                      83.838   
                  >1000                                         54.922   

                              measured_neighbourhood_completeness_pct  \
inference         degree_bin                                            
random_link_split 1                                            99.911   
                  2-5                                          99.896   
                  6-20                                         99.566   
                  21-100                                       95.403   
                  101-1000                                     83.956   
                  >1000                                        55.638   
zero_shot_split   1                                            99.908   
                  2-5                                          99.900   
                  6-20                                         99.584   
                  21-100                                       95.402   
                  101-1000                                     83.799   
                  >1000                                        54.932   

                              predicted_mean_occurrences  \
inference         degree_bin                               
random_link_split 1                               65.908   
                  2-5                             89.693   
                  6-20                           121.062   
                  21-100                         190.948   
                  101-1000                       339.498   
                  >1000                         2672.505   
zero_shot_split   1                               65.952   
                  2-5                             89.838   
                  6-20                           121.507   
                  21-100                         191.601   
                  101-1000                       338.815   
                  >1000                         2660.476   

                              measured_mean_occurrences  
inference         degree_bin                             
random_link_split 1                              65.844  
                  2-5                            89.581  
                  6-20                          120.810  
                  21-100                        189.716  
                  101-1000                      340.387  
                  >1000                        2663.640  
zero_shot_split   1                              65.909  
                  2-5             

relation


training_triples  \
inference         relation                                       
random_link_split anatomy_protein_present            2524012.0   
                  protein_protein                     533786.0   
                  bioprocess_protein                  240738.0   
                  anatomy_protein_absent               33062.0   
                  molfunc_protein                     115594.0   
                  cellcomp_protein                    138656.0   
                  disease_protein                     133682.0   
                  pathway_protein                      70900.0   
                  drug_protein                         42397.2   
                  phenotype_protein                     5536.0   
                  exposure_protein                      2014.0   
                  drug_drug                          2221622.0   
                  drug_effect                         107704.0   
                  contraindication                     25498.0   
                  indication                            7804.0   
                  off-label use                         2135.0   
                  disease_phenotype_positive          249902.0   
                  phenotype_phenotype                  31148.0   
                  disease_phenotype_negative            1982.0   
                  disease_disease                      53522.0   
                  exposure_disease                      3830.0   
                  bioprocess_bioprocess                87924.0   
                  exposure_bioprocess                   2702.0   
                  molfunc_molfunc                      22566.0   
                  exposure_molfunc                        76.0   
                  cellcomp_cellcomp                     8056.0   
                  exposure_cellcomp                       18.0   
                  exposure_exposure                     3440.0   
                  pathway_pathway                       4214.0   
                  anatomy_anatomy                      23328.0   
zero_shot_split   anatomy_protein_present            2524012.0   
                  protein_protein                     533786.0   
                  bioprocess_protein                  240738.0   
                  anatomy_protein_absent               33062.0   
                  molfunc_protein                     115594.0   
                  cellcomp_protein                    138656.0   
                  disease_protein                     133682.0   
                  pathway_protein                      70900.0   
                  drug_protein                         42397.2   
                  phenotype_protein                     5536.0   
                  exposure_protein                      2014.0   
                  drug_drug                          2221622.0   
                  drug_effect                         107704.0   
                  contraindication                     25900.2   
                  indication                            7821.8   
                  off-label use                         4217.6   
                  disease_phenotype_positive          249902.0   
                  phenotype_phenotype                  31148.0   
                  disease_phenotype_negative            1982.0   
                  disease_disease                      53522.0   
                  exposure_disease                      3830.0   
                  bioprocess_bioprocess                87924.0   
                  exposure_bioprocess                   2702.0   
                  molfunc_molfunc                      22566.0   
                  exposure_molfunc                        76.0   
                  cellcomp_cellcomp                     8056.0   
                  exposure_cellcomp                       18.0   
                  exposure_exposure                     3440.0   
                  pathway_pathway                       4214.0   
                  an

curve


predicted  \
inference         walks_per_node metric                                           
random_link_split 5              directed_pair_coverage_pct              19.871   
                                 undirected_pair_coverage_pct            32.593   
                                 mean_neighbourhood_completeness_pct     82.156   
                                 hub_neighbourhood_completeness_pct      15.723   
                  10             directed_pair_coverage_pct              31.884   
                                 undirected_pair_coverage_pct            49.188   
                                 mean_neighbourhood_completeness_pct     89.702   
                                 hub_neighbourhood_completeness_pct      27.315   
                  30             directed_pair_coverage_pct              59.114   
                                 undirected_pair_coverage_pct            77.438   
                                 mean_neighbourhood_completeness_pct     97.225   
                                 hub_neighbourhood_completeness_pct      55.634   
                  60             directed_pair_coverage_pct              76.623   
                                 undirected_pair_coverage_pct            89.724   
                                 mean_neighbourhood_completeness_pct     99.074   
                                 hub_neighbourhood_completeness_pct      76.196   
                  100            directed_pair_coverage_pct              86.456   
                                 undirected_pair_coverage_pct            95.438   
                                 mean_neighbourhood_completeness_pct     99.622   
                                 hub_neighbourhood_completeness_pct      88.905   
zero_shot_split   5              directed_pair_coverage_pct              19.838   
                                 undirected_pair_coverage_pct            32.518   
                                 mean_neighbourhood_completeness_pct     82.202   
                                 hub_neighbourhood_completeness_pct      15.492   
                  10             directed_pair_coverage_pct              31.812   
                                 undirected_pair_coverage_pct            49.040   
                                 mean_neighbourhood_completeness_pct     89.721   
                                 hub_neighbourhood_completeness_pct      26.920   
                  30             directed_pair_coverage_pct              58.922   
                                 undirected_pair_coverage_pct            77.146   
                                 mean_neighbourhood_completeness_pct     97.212   
                                 hub_neighbourhood_completeness_pct      54.922   
                  60             directed_pair_coverage_pct              76.350   
                                 undirected_pair_coverage_pct            89.428   
                                 mean_neighbourhood_completeness_pct     99.057   
                                 hub_neighbourhood_completeness_pct      75.467   
                  100            directed_pair_coverage_pct              86.167   
                                 undirected_pair_coverage_pct            95.225   
                                 mean_neighbourhood_completeness_pct     99.611   
                                 hub_neighbourhood_completeness_pct      88.373   

                                                                      measured  
inference         walks_per_node metric                                         
random_link_split 5              directed_pair_coverage_pct             19.895  
                                 undirected_pair_coverage_pct           32.508  
                                 mean_neighbourhood_completeness_pct    82.162  
                                 hub_neighbourhood_completeness_pct     15.732  
                  10             directed_pair_coverage_pct             31.880  
                

# Check 3+: are the edges that never appear in the walks recovered by the model?

For the model, an edge that the walks never sample is like a held-out test edge. Here this is checked on exactly the edges lost by sampling. Every evaluated edge (head, relation, tail) gets a **per-edge score** with the trained checkpoint: its filtered rank among all candidate tails of its head (all drugs / proteins / effects; the head's other known KG tails are removed from the ranking).
- *AUROC (per edge)*: share of candidate tails scored below the true tail, averaged over edges (0.5 = random).
- *Hits@100*: share of edges whose true tail is in the top 100.

Groups (training edges of the queried relations, plus the test edges):
- **seen**: the two nodes are next to each other in at least one walk;
- **not adjacent, same walk**: never next to each other, but both nodes appear together in at least one walk (at distance ≥ 2);
- **not adjacent, never in same walk**: the two nodes never appear in the same walk at all (no evidence in any pseudo-sentence);
- **not adjacent (all)**: union of the two previous groups;
- **test**: held-out test edges (reference).

**Degree matching.** Every group is re-weighted to the node-degree distribution of the never-walked edges (strata: head degree bin × tail degree bin), so groups are compared at the same node degrees (`*_degree_matched`; `matched_weight_pct` = share of the never-walked edges in strata that the group also has).

Needs a GPU. Output per split / seed: `check3x_edges.csv` (one row per edge); combined: `check3x_edges_all_seeds.csv`, `check3x_summary_all_seeds.csv`.

In [21]:
from netmedgpt.model import TransformerModel

CHECK3_GPU = 0
CHECKPOINT_DIR = "data/model_checkpoints/saved_models_evaluation"
# checkpoint file per split; other splits use the train.py name netmedgpt_{inference}_seed{seed}.pt
CHECKPOINT_NAMES = {"random_link_split": "netmedgpt_randomLinkSplit_seed{seed}.pt",
                    "zero_shot_split": "netmedgpt_zeroShot_seed{seed}.pt"}
CHECK3X_SEEDS = [1, 2, 3, 4, 5]          # seeds (within SEEDS) evaluated
CHECK3X_MAX_EDGES = 5000                 # per relation and group; larger groups are subsampled
CHECK3_HEAD_BATCH = 256                  # heads per forward pass

GROUP_ORDER = ['seen', 'not adjacent (all)', 'not adjacent, same walk', 'not adjacent, never in same walk', 'test']


# NetMedGPT checkpoint of the split / seed, built as in other/evaluate_checkpoints.py
def load_checkpoint(split, device):
    name = CHECKPOINT_NAMES.get(split['inference'], "netmedgpt_{inference}_seed{seed}.pt")
    path = os.path.join(CHECKPOINT_DIR, name.format(inference=split['inference'], seed=split['seed']))
    if not os.path.exists(path):
        print(f"Checkpoint not found: {path}")
        return None

    hp = split['all_param']['best_hyperparam']
    ckpt = torch.load(path, map_location=device)
    if "model_state" in ckpt:
        state_dict, hp, seq_len = ckpt["model_state"], ckpt.get("hyperparams", hp), ckpt.get("seq_len", split['seq_len'])
    else:
        state_dict, seq_len = ckpt, split['seq_len']

    edge, nodes = split['edge'], split['nodes']
    relation_type = list(edge['relation'].unique())
    entity = list(nodes['node_type'].unique()) + relation_type + ['mask']
    relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
    mask_row = pd.DataFrame([['mask', split['mask_token']]], columns=['relation', 'z_index'])
    relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)

    model = TransformerModel(split['vocab_size'], hp['hidden_channels'], hp['nhead'], hp['N_encoder_layers'], seq_len,
                             device=device, feat=split['feat'], nodes=nodes, entity=entity,
                             relation_mask_index=relation_mask_index, pos_emb='fixed').to(device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"Checkpoint loaded: {path}")
    return model


# training edges of one queried relation in query direction (e.g. disease -> drug), flagged seen / unseen in the walks
def training_positives_by_coverage(split, g, cov_und, relation):
    train = split['train_txgnn']
    pos = train.loc[train['relation'] == relation, ['x_index', 'y_index']].drop_duplicates().reset_index(drop=True)
    idx = lookup(g['pairs'], pair_key(pos['x_index'].to_numpy(), pos['y_index'].to_numpy(), g['n_nodes']))
    pos['seen'] = cov_und[g['pair_und_idx'][idx]]
    return pos


# inverted index: for every node, the ids of the walks it appears in
def walk_index(walks, n_nodes):
    node_tok = walks[:, 0::2]
    flat = node_tok.ravel()
    wid = np.repeat(np.arange(len(walks), dtype=np.int64), node_tok.shape[1])
    order = np.argsort(flat, kind='stable')
    indptr = np.r_[0, np.cumsum(np.bincount(flat, minlength=n_nodes))]
    return indptr, wid[order]


# True if the two nodes appear together in at least one walk
def same_walk(index, heads, tails):
    indptr, wid = index
    return np.array([np.intersect1d(wid[indptr[h]:indptr[h + 1]], wid[indptr[t]:indptr[t + 1]]).size > 0
                     for h, t in zip(heads, tails)], dtype=bool)


# all KG tails of a head for one relation in query direction (used to filter the ranking)
def kg_tails(split, relation, node_types):
    e = split['edge']
    e = e[(e['relation'] == relation) & (e['node_types'] == node_types)]
    return e.groupby('x_index')['y_index'].apply(lambda s: np.unique(s.to_numpy())).to_dict()


# filtered rank of every edge among all candidate tails of its head:
#   auc_edge = share of candidates scored below the true tail (other known KG tails removed, ties count half)
#   hit100   = true tail within the top 100
# score_fn(heads) -> numpy array [len(heads), len(cand)]
def edge_ranks(score_fn, edges, cand, known, head_batch):
    cand_np = cand.cpu().numpy()
    col = {int(c): i for i, c in enumerate(cand_np)}
    by_head = edges.groupby('head').indices
    heads = np.array(list(by_head))
    tails = edges['tail'].to_numpy()
    auc = np.full(len(edges), np.nan)
    hit = np.full(len(edges), np.nan)
    for i in range(0, len(heads), head_batch):
        hb = heads[i:i + head_batch]
        S = score_fn(hb)
        for j, h in enumerate(hb):
            row = S[j]
            k = np.array([col[int(t)] for t in known.get(h, []) if int(t) in col], dtype=np.int64)
            for p in by_head[h]:
                c = col.get(int(tails[p]))
                if c is None:
                    continue
                s = row[c]
                kk = k[k != c]
                above = (row > s).sum() - (row[kk] > s).sum()
                ties = (row == s).sum() - 1 - (row[kk] == s).sum()
                rank0 = above + 0.5 * ties
                auc[p] = 1 - rank0 / (len(row) - 1 - len(kk))
                hit[p] = float(rank0 < 100)
    return auc, hit


def netmedgpt_score_fn(model, split, z, cand, device):
    cand_dev = cand.to(device)

    def fn(hb):
        paths = torch.full((len(hb), split['seq_len']), int(split['mask_token']), dtype=torch.long, device=device)
        paths[:, 0] = torch.as_tensor(hb, dtype=torch.long, device=device)
        paths[:, 1] = z
        with torch.no_grad():
            return model(paths)[:, 2, :][:, cand_dev].float().cpu().numpy()
    return fn


def check3x_one_run(split, g, data, st, model, device, rng):
    edge = split['edge']
    cov_und = covered_undirected(g, st)
    index = walk_index(data['node2vec_walks'].numpy(), g['n_nodes'])
    test_idx = data['test_data_edge_label_index_with_relation']
    test_pos = data['test_data_edge_label'] == 1
    test_flag = data['edge_type_flag_test']

    edges_by_rel = {}
    for r, relation in enumerate(split['queried_edge_types']):
        head_type, tail_type = split['queried_node_types'][r].split('|')

        # only edges in query direction (head type -> tail type)
        def query_direction(df):
            return df[(g['node_type'][df['x_index'].to_numpy()] == head_type)
                      & (g['node_type'][df['y_index'].to_numpy()] == tail_type)]

        pos = query_direction(training_positives_by_coverage(split, g, cov_und, relation))
        unseen = pos[~pos['seen']]
        m = test_pos & (test_flag == r + 1)
        test = query_direction(pd.DataFrame({'x_index': test_idx[0, m].numpy(), 'y_index': test_idx[2, m].numpy()}))

        parts = []
        for name, df in [('seen', pos[pos['seen']]), ('not adjacent', unseen), ('test', test)]:
            if len(df) > CHECK3X_MAX_EDGES:
                df = df.iloc[np.sort(rng.choice(len(df), CHECK3X_MAX_EDGES, replace=False))]
            parts.append(pd.DataFrame({'group': name, 'head': df['x_index'].to_numpy(), 'tail': df['y_index'].to_numpy()}))
        e = pd.concat(parts, ignore_index=True)
        na = (e['group'] == 'not adjacent').to_numpy()
        sw = same_walk(index, e.loc[na, 'head'].to_numpy(), e.loc[na, 'tail'].to_numpy())
        e.loc[na, 'group'] = np.where(sw, 'not adjacent, same walk', 'not adjacent, never in same walk')
        e['head_degree'] = g['degree'][e['head'].to_numpy()]
        e['tail_degree'] = g['degree'][e['tail'].to_numpy()]
        edges_by_rel[relation] = e
    del index

    out = []
    for r, relation in enumerate(split['queried_edge_types']):
        z = int(edge.loc[edge['relation'] == relation, 'z_index'].unique().item())
        cand = split['all_ids_node2'][z]
        known = kg_tails(split, relation, split['queried_node_types'][r])
        e = edges_by_rel[relation]
        auc, hit = edge_ranks(netmedgpt_score_fn(model, split, z, cand, device), e, cand, known, CHECK3_HEAD_BATCH)
        res = e.assign(relation=relation, method='NetMedGPT', auc_edge=auc, hit100=hit)
        out.append(res)
        print(f"{relation}\n{res.groupby('group', sort=False)[['auc_edge', 'hit100']].mean().round(4).to_string()}")
    return pd.concat(out, ignore_index=True)


# mean per-edge AUROC and Hits@100 per group, raw and degree-matched to the never-walked edges
def check3x_summary(edges):
    e = edges.copy()
    e['stratum'] = (pd.cut(e['head_degree'], bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS).astype(str) + '|'
                    + pd.cut(e['tail_degree'], bins=DEGREE_BINS, right=False, labels=DEGREE_LABELS).astype(str))
    na = e[e['group'].str.startswith('not adjacent')].assign(group='not adjacent (all)')
    e = pd.concat([e, na], ignore_index=True)

    keys = ['inference', 'seed', 'relation', 'method']
    rows = []
    for k, d in e.groupby(keys, sort=False):
        ref = d.loc[d['group'] == 'not adjacent (all)', 'stratum'].value_counts(normalize=True)
        for group, dg in d.groupby('group', sort=False):
            sm = dg.groupby('stratum')[['auc_edge', 'hit100']].mean()
            common = ref.index.intersection(sm.index)
            w = ref[common]
            matched = sm.loc[common].mul(w, axis=0).sum() / w.sum() if w.sum() > 0 else pd.Series(np.nan, index=sm.columns)
            rows.append({**dict(zip(keys, k)), 'group': group, 'edges': int(dg['auc_edge'].notna().sum()),
                         'auroc': dg['auc_edge'].mean(), 'hits100': dg['hit100'].mean(),
                         'auroc_degree_matched': matched['auc_edge'], 'hits100_degree_matched': matched['hit100'],
                         'matched_weight_pct': 100 * w.sum()})
    out = pd.DataFrame(rows)
    out['group'] = pd.Categorical(out['group'], categories=GROUP_ORDER, ordered=True)
    return out.sort_values(keys + ['group']).reset_index(drop=True)


## Check 3+: run

In [22]:
device = torch.device(f'cuda:{CHECK3_GPU}' if torch.cuda.is_available() else 'cpu')

for inference in INFERENCES:
    for seed in [s for s in SEEDS if s in CHECK3X_SEEDS]:
        print(f"\n===== Check 3+ | {inference} | seed {seed} =====")
        split = load_split(inference, seed, load_features=True)
        model = load_checkpoint(split, device)
        if model is None:
            continue
        data = build_corpus(split)
        g = build_graph(split)
        walks = data['node2vec_walks'].numpy()
        st = WalkStats(g)
        for i in range(0, len(walks), CHUNK):
            st.add(walks[i:i + CHUNK])

        edges = check3x_one_run(split, g, data, st, model, device, np.random.default_rng(seed))

        run_dir = os.path.join(OUT_DIR, f"{inference}_seed{seed}")
        os.makedirs(run_dir, exist_ok=True)
        edges.to_csv(os.path.join(run_dir, "check3x_edges.csv"), index=False)

        del split, model, data, g, st, walks, edges
        torch.cuda.empty_cache()

# combine every split / seed folder that has check 3+ results
parts = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "*_seed*", "check3x_edges.csv"))):
    m = re.match(r"(.+)_seed(\d+)$", os.path.basename(os.path.dirname(path)))
    parts.append(pd.read_csv(path).assign(inference=m.group(1), seed=int(m.group(2))))
if parts:
    check3x_edges = pd.concat(parts, ignore_index=True)
    check3x_edges.to_csv(os.path.join(OUT_DIR, "check3x_edges_all_seeds.csv"), index=False)
    check3x_summary(check3x_edges).to_csv(os.path.join(OUT_DIR, "check3x_summary_all_seeds.csv"), index=False)
    print(f"Check 3+ saved to {OUT_DIR}")


===== Check 3+ | random_link_split | seed 1 =====
Reproducibility settings
  seed                     : 1
  torch.manual_seed        : 1
  torch.cuda.manual_seed   : 1
  numpy / random seed      : 1
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Checkpoint loaded: data/model_checkpoints/saved_models_evaluation/netmedgpt_randomLinkSplit_seed1.pt
Loading cached corpus: data/new_analysis/corpus/random_link_split_seed1.pt
Baselines: 20028 never-walked edges removed from the classifier training pairs
node2vec baselines skipped (No module named 'gensim'); use the walk_baselines environment
DREAMwalk baseline skipped (No module named 'xgboost'); use the walk_baselines environment
indication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9978  0.9618
not adjacent, same walk             0.9867  0.7576
not adjacent, never in same walk    0.9290  0.6720
test                                0.9693  0.8316
contraindication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9941  0.8724
not adjacent, never in same walk    0.9757  0.5792
no

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Checkpoint loaded: data/model_checkpoints/saved_models_evaluation/netmedgpt_randomLinkSplit_seed2.pt
Loading cached corpus: data/new_analysis/corpus/random_link_split_seed2.pt
indication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9979  0.9682
not adjacent, never in same walk    0.9368  0.6684
not adjacent, same walk             0.9913  0.7941
test                                0.9763  0.8252
contraindication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9943  0.8750
not adjacent, never in same walk    0.9734  0.5803
not adjacent, same walk             0.9840  0.6701
test                                0.9823  0.6682
off-label use | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                            

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


... (11 outputs left out)

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Checkpoint loaded: data/model_checkpoints/saved_models_evaluation/netmedgpt_zeroShot_seed4.pt
Loading cached corpus: data/new_analysis/corpus/zero_shot_split_seed4.pt
indication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9962  0.9466
not adjacent, never in same walk    0.9196  0.6460
not adjacent, same walk             0.9821  0.7108
test                                0.9869  0.8546
contraindication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9916  0.8098
not adjacent, same walk             0.9837  0.6071
not adjacent, never in same walk    0.9738  0.5362
test                                0.9767  0.5556

===== Check 3+ | zero_shot_split | seed 5 =====
Reproducibility settings
  seed                     : 5
  torch.manual_seed        : 5
  torch.cuda.manual_seed   : 5
  

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Checkpoint loaded: data/model_checkpoints/saved_models_evaluation/netmedgpt_zeroShot_seed5.pt
Loading cached corpus: data/new_analysis/corpus/zero_shot_split_seed5.pt
indication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9974  0.9562
not adjacent, never in same walk    0.9323  0.6394
not adjacent, same walk             0.9684  0.5529
test                                0.9622  0.7427
contraindication | NetMedGPT
                                  auc_edge  hit100
group                                             
seen                                0.9939  0.8642
not adjacent, never in same walk    0.9770  0.5858
not adjacent, same walk             0.9828  0.6091
test                                0.9595  0.4532
Check 3+ saved to data/result/new_analysis/walk_coverage


## Check 3+: results (mean over seeds)

In [23]:
summary3x = pd.read_csv(os.path.join(OUT_DIR, "check3x_summary_all_seeds.csv"))
summary3x['group'] = pd.Categorical(summary3x['group'], categories=GROUP_ORDER, ordered=True)
metrics = ['edges', 'auroc', 'auroc_degree_matched', 'hits100', 'hits100_degree_matched', 'matched_weight_pct']

nm = summary3x[summary3x['method'] == 'NetMedGPT']
display(nm.groupby(['inference', 'relation', 'group'], sort=False, observed=True)[metrics].mean().round(4))


a/b) NetMedGPT, all seeds


edges  \
inference         relation         group                                      
random_link_split contraindication seen                              5000.0   
                                   not adjacent (all)                5000.0   
                                   not adjacent, same walk            373.4   
                                   not adjacent, never in same walk  4626.6   
                                   test                              1534.0   
                  drug_effect      seen                              5000.0   
                                   not adjacent (all)                5000.0   
                                   not adjacent, same walk           1036.0   
                                   not adjacent, never in same walk  3964.0   
                                   test                              3239.0   
                  drug_protein     seen                              5000.0   
                                   not adjacent (all)                2312.0   
                                   not adjacent, same walk            381.6   
                                   not adjacent, never in same walk  1930.4   
                                   test                              1283.0   
                  indication       seen                              5000.0   
                                   not adjacent (all)                1027.2   
                                   not adjacent, same walk             70.4   
                                   not adjacent, never in same walk   956.8   
                                   test                               469.0   
                  off-label use    seen                              1962.2   
                                   not adjacent (all)                 172.8   
                                   not adjacent, same walk             11.2   
                                   not adjacent, never in same walk   161.6   
                                   test                               128.0   
zero_shot_split   contraindication seen                              5000.0   
                                   not adjacent (all)                5000.0   
                                   not adjacent, same walk            394.6   
                                   not adjacent, never in same walk  4605.4   
                                   test                              1133.2   
                  indication       seen                              5000.0   
                                   not adjacent (all)                1146.2   
                                   not adjacent, same walk             75.4   
                                   not adjacent, never in same walk  1070.8   
                                   test                               465.0   

                                                                      auroc  \
inference         relation         group                                      
random_link_split contraindication seen                              0.9941   
                                   not adjacent (all)                0.9750   
                                   not adjacent, same walk           0.9836   
                                   not adjacent, never in same walk  0.9743   
                                   test                              0.9827   
                  drug_effect      seen                              0.9972   
                                   not adjacent (all)                0.9977   
                                   not adjacent, same walk           0.9985   
                                   not adjacent, never in same walk  0.9974   
                                   test                              0.9956   
                  drug_protein     seen                              0.9981   
                                   not adjacent (all)                0.9867   
                                   not a

c) NetMedGPT vs walk baselines on the same edges (seeds with baselines)


In [27]:
print

<function print>